# 10 - Frozen-Backbone LSTM Feature Selection Screen - 22.03.2026

This notebook is the notebook-first replacement for the separate experiment repo.

The question is narrow on purpose:

`With the LSTM backbone frozen, which temporal feature blocks actually improve direct point-horizon heat-demand forecasting?`

Why this notebook exists:

- feature selection is easier to inspect in notebook form,
- the code, design, tables, and interpretation all live together,
- it fits the current thesis workflow better than a separate package.

This notebook deliberately focuses on **temporal `setA` features** only.
The richer `setB` static and `stat_train_*` fields are left for a later, leak-checked follow-up experiment.

In [1]:
from __future__ import annotations

import json
import os
import sys
import types
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_ROOT = Path('/Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project')
FEATURE_METADATA_FILE = PROJECT_ROOT / 'data' / 'features' / 'feature_metadata.csv'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'feature_selection_screen_22032026'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_TARGET_SOURCE_COL = 'heat_kwh'
DEFAULT_FEATURE_SET = 'setA'
PRIMARY_METRIC = 'wape_heating_pct'

SCREEN_BUILDINGS = ['U05', 'U06']
SCREEN_HORIZONS = [1, 3, 4, 6, 8]
SCREEN_MONTHS = [1, 2, 3, 11, 12]

HEATING_TEMP_THRESHOLD_C = 15.0
SEED = 42
EPOCHS = 18
BATCH_SIZE = 64
EARLY_STOPPING_PATIENCE = 4
FIT_VALIDATION_FRACTION = 0.10
MAX_TRAIN_SEQUENCES = 25000

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## Frozen Backbone

The architecture is intentionally fixed so the experiment answers a feature question, not an architecture question.

In [3]:
@dataclass(frozen=True)
class FrozenArchitecture:
    key: str
    label: str
    lookback_hours: int
    lstm_units: tuple[int, ...]
    dropout: float
    dense_units: int


ARCHITECTURE = FrozenArchitecture(
    key='single_64_l72',
    label='single 64 | L72',
    lookback_hours=72,
    lstm_units=(64,),
    dropout=0.0,
    dense_units=16,
)

ARCHITECTURE

FrozenArchitecture(key='single_64_l72', label='single 64 | L72', lookback_hours=72, lstm_units=(64,), dropout=0.0, dense_units=16)

## Feature Blocks

The main idea is to compare **blocks** rather than ranking all columns one by one.
That produces a much cleaner thesis story.

In [4]:
BLOCKS: dict[str, list[str]] = {
    'core': [
        'feat_heat_obs',
        'feat_outdoor_temp_c',
        'feat_hour_sin',
        'feat_hour_cos',
        'feat_dow_sin',
        'feat_dow_cos',
        'feat_is_weekend',
    ],
    'heat_history': [
        'feat_heat_lag1',
        'feat_heat_lag3',
        'feat_heat_lag24',
        'feat_heat_lag168',
        'feat_heat_roll6h',
        'feat_heat_roll24h',
    ],
    'temp_memory': [
        'feat_hdh_15c',
        'feat_temp_roll6h',
        'feat_temp_roll24h',
        'feat_temp_diff1h',
        'feat_temp_diff3h',
    ],
    'space_loop': [
        'feat_space_heat_active',
        'feat_space_deltaT_c',
        'feat_space_deltaT_lag1',
        'feat_space_deltaT_roll6h',
        'feat_space_low_deltaT_flag',
    ],
    'vent_loop': [
        'feat_vent_heat_active',
        'feat_vent_deltaT_c',
        'feat_vent_deltaT_lag1',
        'feat_vent_deltaT_roll6h',
        'feat_vent_low_deltaT_flag',
    ],
    'dhw_loop': [
        'feat_dhw_heat_active',
        'feat_dhw_deltaT_c',
        'feat_dhw_deltaT_lag1',
        'feat_dhw_deltaT_roll6h',
        'feat_dhw_low_deltaT_flag',
    ],
    'weather_extras': [
        'feat_wind_ms',
        'feat_rh_pct',
        'feat_windchill_c',
    ],
    'solar': [
        'feat_solar_irradiance_wm2',
        'feat_sunshine_min',
    ],
}

VARIANTS = {
    'V00_core': ('core',),
    'V10_core_heat': ('core', 'heat_history'),
    'V11_core_temp': ('core', 'temp_memory'),
    'V12_core_space': ('core', 'space_loop'),
    'V13_core_vent': ('core', 'vent_loop'),
    'V14_core_dhw': ('core', 'dhw_loop'),
    'V15_core_weather': ('core', 'weather_extras'),
    'V16_core_solar': ('core', 'solar'),
    'V20_core_heat_temp': ('core', 'heat_history', 'temp_memory'),
    'V21_core_heat_temp_space': ('core', 'heat_history', 'temp_memory', 'space_loop'),
    'V22_core_heat_temp_vent': ('core', 'heat_history', 'temp_memory', 'vent_loop'),
    'V23_core_heat_temp_space_vent': ('core', 'heat_history', 'temp_memory', 'space_loop', 'vent_loop'),
    'V24_all_dynamic_no_solar': ('core', 'heat_history', 'temp_memory', 'space_loop', 'vent_loop', 'dhw_loop', 'weather_extras'),
    'V25_all_dynamic_with_solar': ('core', 'heat_history', 'temp_memory', 'space_loop', 'vent_loop', 'dhw_loop', 'weather_extras', 'solar'),
}


def variant_columns(variant_id: str) -> list[str]:
    cols: list[str] = []
    seen: set[str] = set()
    for block_name in VARIANTS[variant_id]:
        for col in BLOCKS[block_name]:
            if col not in seen:
                cols.append(col)
                seen.add(col)
    return cols


variant_overview = pd.DataFrame([
    {
        'variant_id': variant_id,
        'blocks': ', '.join(VARIANTS[variant_id]),
        'n_features': len(variant_columns(variant_id)),
    }
    for variant_id in VARIANTS
])

display(variant_overview)

,variant_id,blocks,n_features
0,V00_core,core,7
1,V10_core_heat,"core, heat_history",13
2,V11_core_temp,"core, temp_memory",12
3,V12_core_space,"core, space_loop",12
4,V13_core_vent,"core, vent_loop",12
5,V14_core_dhw,"core, dhw_loop",12
6,V15_core_weather,"core, weather_extras",10
7,V16_core_solar,"core, solar",9
8,V20_core_heat_temp,"core, heat_history, temp_memory",18
9,V21_core_heat_temp_space,"core, heat_history, temp_memory, space_loop",23


## Data Helpers

In [5]:
def load_feature_metadata() -> pd.DataFrame:
    meta = pd.read_csv(FEATURE_METADATA_FILE)
    return meta


def feature_path_for_building(building: str, set_name: str = DEFAULT_FEATURE_SET) -> Path:
    meta = load_feature_metadata()
    row = meta.loc[meta['building'] == building]
    if row.empty:
        raise KeyError(f'Building {building} not found in feature metadata')
    rel = row.iloc[0][f'path_{set_name}']
    return (PROJECT_ROOT / str(rel)).resolve()


def load_feature_frame(building: str, set_name: str = DEFAULT_FEATURE_SET) -> pd.DataFrame:
    path = feature_path_for_building(building, set_name=set_name)
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df.sort_values('datetime').reset_index(drop=True)


def add_direct_target(df: pd.DataFrame, horizon_hours: int, source_col: str = DEFAULT_TARGET_SOURCE_COL) -> tuple[pd.DataFrame, str]:
    out = df.copy()
    target_col = f'target_point_h{int(horizon_hours)}'
    out[target_col] = out[source_col].shift(-int(horizon_hours))
    return out, target_col


def heating_mask(df: pd.DataFrame) -> pd.Series:
    if 'feat_is_heating_weather' in df.columns:
        return df['feat_is_heating_weather'].fillna(0).astype(float) > 0.5
    return df['feat_outdoor_temp_c'].astype(float) < HEATING_TEMP_THRESHOLD_C

## Time-Aware Fold Design

In [6]:
@dataclass(frozen=True)
class MonthlyFold:
    fold_id: str
    train_end: pd.Timestamp
    test_start: pd.Timestamp
    test_end: pd.Timestamp


def build_monthly_folds(year: int, months: list[int]) -> list[MonthlyFold]:
    folds = []
    for month in months:
        test_start = pd.Timestamp(year=year, month=int(month), day=1, hour=0)
        test_end = (test_start + pd.offsets.MonthEnd(1)).replace(hour=23)
        train_end = test_start - pd.Timedelta(hours=1)
        folds.append(MonthlyFold(
            fold_id=f'{test_start:%Y-%m}',
            train_end=train_end,
            test_start=test_start,
            test_end=test_end,
        ))
    return folds


FOLDS = build_monthly_folds(2024, SCREEN_MONTHS)
pd.DataFrame([fold.__dict__ for fold in FOLDS])

,fold_id,train_end,test_start,test_end
0,2024-01,2023-12-31 23:00:00,2024-01-01,2024-01-31 23:00:00
1,2024-02,2024-01-31 23:00:00,2024-02-01,2024-02-29 23:00:00
2,2024-03,2024-02-29 23:00:00,2024-03-01,2024-03-31 23:00:00
3,2024-11,2024-10-31 23:00:00,2024-11-01,2024-11-30 23:00:00
4,2024-12,2024-11-30 23:00:00,2024-12-01,2024-12-31 23:00:00


## TensorFlow Import Helper

This machine has a local `cv2` packaging quirk that Keras can trip over during import.
The small stub below avoids that path for this notebook.

In [7]:
def install_cv2_stub() -> None:
    module = sys.modules.get('cv2')
    if module is None:
        module = types.ModuleType('cv2')
        module.dnn = types.SimpleNamespace(DictValue=object)
        sys.modules['cv2'] = module
        return

    if not hasattr(module, 'dnn'):
        module.dnn = types.SimpleNamespace(DictValue=object)
        return

    if not hasattr(module.dnn, 'DictValue'):
        module.dnn.DictValue = object


def import_tf():
    os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
    os.environ.setdefault('MPLCONFIGDIR', '/tmp/codex-mplconfig')
    os.environ.setdefault('XDG_CACHE_HOME', '/tmp')
    install_cv2_stub()
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    tf.get_logger().setLevel('ERROR')
    return tf, keras, layers


def set_all_seeds(seed: int = SEED) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

## Frozen LSTM Builder

In [8]:
def build_lstm_model(n_features: int, architecture: FrozenArchitecture = ARCHITECTURE):
    _, keras, layers = import_tf()

    temporal_in = keras.Input(shape=(architecture.lookback_hours, n_features), name='temporal_input')
    x = temporal_in
    for idx, units in enumerate(architecture.lstm_units):
        return_sequences = idx < len(architecture.lstm_units) - 1
        x = layers.LSTM(units, return_sequences=return_sequences, name=f'lstm_{idx + 1}')(x)
        if return_sequences and architecture.dropout > 0:
            x = layers.Dropout(architecture.dropout, name=f'dropout_{idx + 1}')(x)
    x = layers.Dense(architecture.dense_units, activation='relu', name='dense_1')(x)
    out = layers.Dense(1, name='output')(x)

    try:
        optimizer = keras.optimizers.legacy.Adam(learning_rate=1e-3)
    except Exception:
        optimizer = keras.optimizers.Adam(learning_rate=1e-3)

    model = keras.Model(inputs=temporal_in, outputs=out, name=architecture.key)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mse'])
    return model


def clear_tf_session() -> None:
    _, keras, _ = import_tf()
    keras.backend.clear_session()

## Sequence Prep and Metrics

In [9]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    if len(y_true) == 0:
        return {'mae': np.nan, 'rmse': np.nan, 'wape_pct': np.nan, 'r2': np.nan}
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    denom = float(np.sum(np.abs(y_true)))
    wape = float(100.0 * np.sum(np.abs(y_true - y_pred)) / denom) if denom > 0 else np.nan
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan
    return {'mae': mae, 'rmse': rmse, 'wape_pct': wape, 'r2': r2}


def prepare_scaled_arrays(df: pd.DataFrame, feature_cols: list[str], target_col: str, train_end: pd.Timestamp, issue_train_end: pd.Timestamp):
    dt = pd.to_datetime(df['datetime'])

    imputer = SimpleImputer(strategy='median')
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()

    fit_x = df.loc[dt <= train_end, feature_cols]
    fit_y = df.loc[(dt <= issue_train_end) & df[target_col].notna(), [target_col]]
    if fit_x.empty or fit_y.empty:
        raise ValueError('No train rows available for feature or target scaling')

    scaler_x.fit(imputer.fit_transform(fit_x))
    scaler_y.fit(fit_y.values)

    X_all = scaler_x.transform(imputer.transform(df[feature_cols]))
    y_all = np.full(len(df), np.nan, dtype='float32')
    valid_y = df[target_col].notna()
    if valid_y.any():
        y_all[valid_y.to_numpy()] = scaler_y.transform(df.loc[valid_y, [target_col]].values)[:, 0]

    return X_all, y_all, scaler_y, dt.to_numpy()


def build_sequences(features, target_scaled, timestamps, heating_flags, issue_mask, lookback, max_sequences=None):
    X_list, y_list, ts_list, heat_list = [], [], [], []
    for end_idx in range(lookback, len(features)):
        if not issue_mask[end_idx]:
            continue
        target_val = target_scaled[end_idx]
        if np.isnan(target_val):
            continue
        window = features[end_idx - lookback:end_idx]
        if np.isnan(window).any():
            continue
        X_list.append(window.astype('float32'))
        y_list.append(float(target_val))
        ts_list.append(np.datetime64(pd.Timestamp(timestamps[end_idx])))
        heat_list.append(bool(heating_flags[end_idx]))

    if not X_list:
        return (
            np.empty((0, lookback, features.shape[1]), dtype='float32'),
            np.empty((0,), dtype='float32'),
            np.empty((0,), dtype='datetime64[ns]'),
            np.empty((0,), dtype=bool),
        )

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype='float32')
    ts = np.array(ts_list, dtype='datetime64[ns]')
    heat = np.array(heat_list, dtype=bool)

    if max_sequences and len(X) > max_sequences:
        X = X[-max_sequences:]
        y = y[-max_sequences:]
        ts = ts[-max_sequences:]
        heat = heat[-max_sequences:]

    return X, y, ts, heat


def split_train_validation(X, y, heat_flags, frac=FIT_VALIDATION_FRACTION):
    if len(X) < 2 or frac <= 0:
        empty_x = np.empty((0,) + X.shape[1:], dtype=X.dtype)
        empty_y = np.empty((0,), dtype=y.dtype)
        empty_h = np.empty((0,), dtype=bool)
        return X, y, heat_flags, empty_x, empty_y, empty_h

    n_val = max(1, int(round(len(X) * frac)))
    n_val = min(n_val, len(X) - 1)
    split_at = len(X) - n_val
    return (
        X[:split_at], y[:split_at], heat_flags[:split_at],
        X[split_at:], y[split_at:], heat_flags[split_at:]
    )

## Single Run Function

In [10]:
def run_single_training(df: pd.DataFrame, building: str, variant_id: str, horizon_hours: int, fold: MonthlyFold) -> dict[str, object]:
    feature_cols = variant_columns(variant_id)
    missing_cols = [col for col in feature_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f'Missing columns for {building} {variant_id}: {missing_cols}')

    work_df, target_col = add_direct_target(df, horizon_hours=horizon_hours, source_col=DEFAULT_TARGET_SOURCE_COL)
    heat_flags = heating_mask(work_df).to_numpy(dtype=bool)

    issue_train_end = fold.train_end - pd.Timedelta(hours=int(horizon_hours))
    issue_test_end = fold.test_end - pd.Timedelta(hours=int(horizon_hours))

    X_all, y_all, scaler_y, timestamps = prepare_scaled_arrays(
        work_df,
        feature_cols=feature_cols,
        target_col=target_col,
        train_end=fold.train_end,
        issue_train_end=issue_train_end,
    )

    dt = pd.to_datetime(work_df['datetime'])
    train_issue_mask = (dt <= issue_train_end).to_numpy(dtype=bool)
    test_issue_mask = ((dt >= fold.test_start) & (dt <= issue_test_end)).to_numpy(dtype=bool)

    X_train_all, y_train_all, _, heat_train_all = build_sequences(
        X_all, y_all, timestamps, heat_flags, train_issue_mask,
        lookback=ARCHITECTURE.lookback_hours,
        max_sequences=MAX_TRAIN_SEQUENCES,
    )
    X_test, y_test, ts_test, heat_test = build_sequences(
        X_all, y_all, timestamps, heat_flags, test_issue_mask,
        lookback=ARCHITECTURE.lookback_hours,
    )

    base = {
        'status': 'ok',
        'building': building,
        'variant_id': variant_id,
        'horizon_hours': int(horizon_hours),
        'target_name': target_col,
        'feature_count': len(feature_cols),
        'feature_list': ','.join(feature_cols),
        'fold_id': fold.fold_id,
        'train_end': str(fold.train_end),
        'test_start': str(fold.test_start),
        'test_end': str(fold.test_end),
        'architecture_key': ARCHITECTURE.key,
        'architecture_label': ARCHITECTURE.label,
        'lookback_hours': ARCHITECTURE.lookback_hours,
    }

    if len(X_train_all) < 64 or len(X_test) < 24:
        return {
            **base,
            'status': 'skipped',
            'skip_reason': 'insufficient_sequences',
            'n_train_sequences': int(len(X_train_all)),
            'n_test_sequences': int(len(X_test)),
        }

    X_fit, y_fit, _, X_val, y_val, _ = split_train_validation(X_train_all, y_train_all, heat_train_all)

    set_all_seeds(SEED)
    clear_tf_session()
    _, keras, _ = import_tf()

    model = build_lstm_model(n_features=len(feature_cols))
    callbacks = []
    if len(X_val) > 0:
        callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=EARLY_STOPPING_PATIENCE,
                restore_best_weights=True,
            )
        )

    history = model.fit(
        X_fit,
        y_fit,
        validation_data=(X_val, y_val) if len(X_val) > 0 else None,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=callbacks,
    )

    pred_scaled = model.predict(X_test, verbose=0).reshape(-1, 1)
    y_pred = scaler_y.inverse_transform(pred_scaled).reshape(-1)
    y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).reshape(-1)

    metrics_all = compute_metrics(y_true, y_pred)
    metrics_heat = compute_metrics(y_true[heat_test], y_pred[heat_test])

    return {
        **base,
        'n_train_sequences': int(len(X_train_all)),
        'n_validation_sequences': int(len(X_val)),
        'n_test_sequences': int(len(X_test)),
        'n_test_heating_sequences': int(heat_test.sum()),
        'epochs_ran': int(len(history.history.get('loss', []))),
        'best_val_loss': float(np.nanmin(history.history.get('val_loss', [np.nan]))),
        'mae': metrics_all['mae'],
        'rmse': metrics_all['rmse'],
        'wape_pct': metrics_all['wape_pct'],
        'r2': metrics_all['r2'],
        'mae_heating': metrics_heat['mae'],
        'rmse_heating': metrics_heat['rmse'],
        'wape_heating_pct': metrics_heat['wape_pct'],
        'r2_heating': metrics_heat['r2'],
        'test_datetime_start': str(pd.Timestamp(ts_test[0])) if len(ts_test) else None,
        'test_datetime_end': str(pd.Timestamp(ts_test[-1])) if len(ts_test) else None,
    }

## Experiment Runner and Selection Rule

In [11]:
def summarize_results(run_log: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, object]]:
    ok = run_log.loc[run_log['status'] == 'ok'].copy()
    if ok.empty:
        raise ValueError('No successful runs to summarize')

    core = ok.loc[ok['variant_id'] == 'V00_core', ['building', 'horizon_hours', 'fold_id', PRIMARY_METRIC]].copy()
    core = core.rename(columns={PRIMARY_METRIC: 'core_wape_heating_pct'})
    merged = ok.merge(core, on=['building', 'horizon_hours', 'fold_id'], how='left')
    merged['delta_vs_core_pp'] = merged[PRIMARY_METRIC] - merged['core_wape_heating_pct']
    merged['win_vs_core'] = merged['delta_vs_core_pp'] < 0

    summary = (
        merged.groupby('variant_id', as_index=False)
        .agg(
            feature_count=('feature_count', 'first'),
            n_runs=('variant_id', 'size'),
            mean_wape_heating_pct=('wape_heating_pct', 'mean'),
            std_wape_heating_pct=('wape_heating_pct', 'std'),
            mean_rmse_heating=('rmse_heating', 'mean'),
            mean_mae_heating=('mae_heating', 'mean'),
            mean_r2_heating=('r2_heating', 'mean'),
            mean_delta_vs_core_pp=('delta_vs_core_pp', 'mean'),
            median_delta_vs_core_pp=('delta_vs_core_pp', 'median'),
            win_rate_vs_core=('win_vs_core', 'mean'),
        )
        .sort_values(['mean_wape_heating_pct', 'feature_count', 'variant_id'])
        .reset_index(drop=True)
    )

    best_mean = float(summary['mean_wape_heating_pct'].min())
    promoted = summary.loc[
        (summary['variant_id'] != 'V00_core')
        & (summary['mean_delta_vs_core_pp'] <= -0.3)
        & (summary['win_rate_vs_core'] >= 0.55)
    ].copy()

    if promoted.empty:
        pool = summary.loc[summary['mean_wape_heating_pct'] <= best_mean + 0.3].copy()
        selection_rule = 'fallback_simplest_within_0p3_of_best'
    else:
        pool = promoted
        selection_rule = 'promoted_by_delta_and_win_rate'

    selected = pool.sort_values(['mean_wape_heating_pct', 'feature_count', 'variant_id']).iloc[0]
    selection = {
        'primary_metric': PRIMARY_METRIC,
        'selection_rule': selection_rule,
        'selected_variant_id': selected['variant_id'],
        'selected_feature_count': int(selected['feature_count']),
        'selected_mean_wape_heating_pct': float(selected['mean_wape_heating_pct']),
        'selected_mean_delta_vs_core_pp': float(selected['mean_delta_vs_core_pp']),
        'selected_win_rate_vs_core': float(selected['win_rate_vs_core']),
        'best_mean_wape_heating_pct': best_mean,
    }
    return summary, selection


def run_screen(
    buildings: list[str] = SCREEN_BUILDINGS,
    horizons: list[int] = SCREEN_HORIZONS,
    months: list[int] = SCREEN_MONTHS,
    variant_ids: list[str] | None = None,
    year: int = 2024,
    results_subdir: str = 'screen',
    verbose: bool = True,
    save_every: int = 5,
):
    if variant_ids is None:
        variant_ids = list(VARIANTS.keys())

    folds = build_monthly_folds(year, months)
    fold_ids = {fold.fold_id for fold in folds}
    data_cache = {building: load_feature_frame(building, set_name=DEFAULT_FEATURE_SET) for building in buildings}
    total_runs = len(buildings) * len(horizons) * len(variant_ids) * len(folds)

    results_path = RESULTS_DIR / results_subdir
    results_path.mkdir(parents=True, exist_ok=True)
    run_log_path = results_path / 'run_log.csv'

    rows = []
    completed_keys: set[tuple[str, int, str, str]] = set()
    if run_log_path.exists():
        existing_run_log = pd.read_csv(run_log_path)
        if not existing_run_log.empty:
            required_cols = ['building', 'horizon_hours', 'variant_id', 'fold_id']
            if all(col in existing_run_log.columns for col in required_cols):
                existing_run_log = existing_run_log.loc[
                    existing_run_log['building'].isin(buildings)
                    & existing_run_log['horizon_hours'].isin(horizons)
                    & existing_run_log['variant_id'].isin(variant_ids)
                    & existing_run_log['fold_id'].isin(fold_ids)
                ].copy()
                existing_run_log['horizon_hours'] = existing_run_log['horizon_hours'].astype(int)
                existing_run_log = existing_run_log.drop_duplicates(
                    subset=['building', 'horizon_hours', 'variant_id', 'fold_id'],
                    keep='last',
                )
                rows = existing_run_log.to_dict('records')
                completed_keys = {
                    (str(row['building']), int(row['horizon_hours']), str(row['variant_id']), str(row['fold_id']))
                    for row in rows
                }

    completed = len(completed_keys)

    if verbose:
        print(
            f'Starting feature screen: {total_runs} runs '
            f'({len(buildings)} buildings x {len(horizons)} horizons x {len(variant_ids)} variants x {len(folds)} folds)'
        )
        print(f'Results will be written to: {results_path}')
        if completed > 0:
            print(f'Resuming from existing run_log.csv: {completed}/{total_runs} runs already completed')

    for building in buildings:
        df = data_cache[building]
        for horizon in horizons:
            for variant_id in variant_ids:
                for fold in folds:
                    run_key = (building, int(horizon), variant_id, fold.fold_id)
                    if run_key in completed_keys:
                        continue

                    started_at = perf_counter()
                    try:
                        row = run_single_training(df, building, variant_id, horizon, fold)
                    except Exception as exc:
                        row = {
                            'status': 'error',
                            'building': building,
                            'variant_id': variant_id,
                            'horizon_hours': int(horizon),
                            'fold_id': fold.fold_id,
                            'architecture_key': ARCHITECTURE.key,
                            'architecture_label': ARCHITECTURE.label,
                            'feature_count': len(variant_columns(variant_id)),
                            'error_type': type(exc).__name__,
                            'error_message': str(exc),
                        }

                    elapsed_s = perf_counter() - started_at
                    completed += 1
                    completed_keys.add(run_key)
                    rows.append(row)

                    if verbose:
                        status = row['status']
                        prefix = f'[{completed:>3}/{total_runs}] {building} | h={int(horizon)} | {variant_id} | {fold.fold_id}'
                        if status == 'ok':
                            metric = row.get(PRIMARY_METRIC)
                            epochs_ran = row.get('epochs_ran')
                            print(
                                f'{prefix} | ok | {PRIMARY_METRIC}={metric:.3f} | '
                                f'epochs={epochs_ran} | {elapsed_s:.1f}s',
                                flush=True,
                            )
                        elif status == 'skipped':
                            print(
                                f'{prefix} | skipped | {row.get("skip_reason", "unknown_reason")} | {elapsed_s:.1f}s',
                                flush=True,
                            )
                        else:
                            print(
                                f'{prefix} | error | {row.get("error_type", "Exception")}: {row.get("error_message", "")} | {elapsed_s:.1f}s',
                                flush=True,
                            )

                    if save_every > 0 and (completed % save_every == 0 or completed == total_runs):
                        pd.DataFrame(rows).to_csv(run_log_path, index=False)

    run_log = pd.DataFrame(rows)
    if not run_log.empty:
        run_log = run_log.drop_duplicates(
            subset=['building', 'horizon_hours', 'variant_id', 'fold_id'],
            keep='last',
        ).reset_index(drop=True)
    run_log.to_csv(run_log_path, index=False)

    ok = run_log.loc[run_log['status'] == 'ok'].copy()
    summary = None
    selection = None
    if not ok.empty:
        summary, selection = summarize_results(run_log)
        summary.to_csv(results_path / 'variant_summary.csv', index=False)
        (results_path / 'selection_summary.json').write_text(json.dumps(selection, indent=2))

    metadata = {
        'architecture': ARCHITECTURE.__dict__,
        'buildings': buildings,
        'horizons': horizons,
        'months': months,
        'variant_ids': variant_ids,
        'results_dir': str(results_path),
    }
    (results_path / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))

    if verbose:
        n_ok = int((run_log['status'] == 'ok').sum())
        n_skipped = int((run_log['status'] == 'skipped').sum())
        n_error = int((run_log['status'] == 'error').sum())
        print(f'Finished feature screen: ok={n_ok}, skipped={n_skipped}, error={n_error}')
        if summary is not None:
            best = summary.iloc[0]
            summary_metric_col = f'mean_{PRIMARY_METRIC}'
            print(
                f'Current best variant: {best["variant_id"]} '
                f'with mean {summary_metric_col}={best[summary_metric_col]:.3f}'
            )

    return run_log, summary, selection

## Smoke Run

Uncomment the next cell if you want a tiny sanity check first.

In [12]:
smoke_run_log, smoke_summary, smoke_selection = run_screen(
    buildings=['U05'],
    horizons=[1],
    months=[1],
    variant_ids=['V00_core', 'V10_core_heat'],
    results_subdir='smoke',
    verbose=True,
    save_every=1,
)
display(smoke_run_log)
display(smoke_summary)
smoke_selection

Starting feature screen: 2 runs (1 buildings x 1 horizons x 2 variants x 1 folds)
Results will be written to: /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/results/feature_selection_screen_22032026/smoke
Resuming from existing run_log.csv: 2/2 runs already completed
Finished feature screen: ok=2, skipped=0, error=0
Current best variant: V10_core_heat with mean mean_wape_heating_pct=19.850


,status,building,variant_id,horizon_hours,target_name,feature_count,feature_list,fold_id,train_end,test_start,test_end,architecture_key,architecture_label,lookback_hours,n_train_sequences,n_validation_sequences,n_test_sequences,n_test_heating_sequences,epochs_ran,best_val_loss,mae,rmse,wape_pct,r2,mae_heating,rmse_heating,wape_heating_pct,r2_heating,test_datetime_start,test_datetime_end
0,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-01,2023-12-31 23:00:00,2024-01-01 00:00:00,2024-01-31 23:00:00,single_64_l72,single 64 | L72,72,16378,1638,743,743,13,0.180841,29.540422,41.016574,20.964905,0.554849,29.540422,41.016574,20.964905,0.554849,2024-01-01 00:00:00,2024-01-31 22:00:00
1,ok,U05,V10_core_heat,1,target_point_h1,13,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-01,2023-12-31 23:00:00,2024-01-01 00:00:00,2024-01-31 23:00:00,single_64_l72,single 64 | L72,72,16378,1638,743,743,5,0.157720,27.969730,37.701723,19.850183,0.623894,27.969730,37.701723,19.850183,0.623894,2024-01-01 00:00:00,2024-01-31 22:00:00


,variant_id,feature_count,n_runs,mean_wape_heating_pct,std_wape_heating_pct,mean_rmse_heating,mean_mae_heating,mean_r2_heating,mean_delta_vs_core_pp,median_delta_vs_core_pp,win_rate_vs_core
0,V10_core_heat,13,1,19.850183,NaN,37.701723,27.969730,0.623894,-1.114721,-1.114721,1.0
1,V00_core,7,1,20.964905,NaN,41.016574,29.540422,0.554849,0.000000,0.000000,0.0


{'primary_metric': 'wape_heating_pct',
 'selection_rule': 'promoted_by_delta_and_win_rate',
 'selected_variant_id': 'V10_core_heat',
 'selected_feature_count': 13,
 'selected_mean_wape_heating_pct': 19.850183486938477,
 'selected_mean_delta_vs_core_pp': -1.1147212982177734,
 'selected_win_rate_vs_core': 1.0,
 'best_mean_wape_heating_pct': 19.850183486938477}

## Main Screen

Run this cell when you want the actual feature screen.

In [13]:
run_log, variant_summary, selection = run_screen(verbose=True, save_every=5)
display(run_log.head())
display(variant_summary)
selection

Starting feature screen: 700 runs (2 buildings x 5 horizons x 14 variants x 5 folds)
Results will be written to: /Users/mihkeluutar/Documents/TalTech/Thesis/thesis-project/results/feature_selection_screen_22032026/screen
Resuming from existing run_log.csv: 390/700 runs already completed


I0000 00:00:1774194831.278526 2010744 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1774194831.278705 2010744 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


[391/700] U06 | h=1 | V20_core_heat_temp | 2024-01 | ok | wape_heating_pct=10.191 | epochs=18 | 69.0s
[392/700] U06 | h=1 | V20_core_heat_temp | 2024-02 | ok | wape_heating_pct=6.932 | epochs=16 | 67.2s
[393/700] U06 | h=1 | V20_core_heat_temp | 2024-03 | ok | wape_heating_pct=10.944 | epochs=12 | 52.2s
[394/700] U06 | h=1 | V20_core_heat_temp | 2024-11 | ok | wape_heating_pct=17.847 | epochs=6 | 34.3s
[395/700] U06 | h=1 | V20_core_heat_temp | 2024-12 | ok | wape_heating_pct=6.355 | epochs=12 | 65.0s
[396/700] U06 | h=1 | V21_core_heat_temp_space | 2024-01 | ok | wape_heating_pct=12.747 | epochs=18 | 66.3s
[397/700] U06 | h=1 | V21_core_heat_temp_space | 2024-02 | ok | wape_heating_pct=6.566 | epochs=13 | 49.7s
[398/700] U06 | h=1 | V21_core_heat_temp_space | 2024-03 | ok | wape_heating_pct=12.324 | epochs=8 | 32.6s
[399/700] U06 | h=1 | V21_core_heat_temp_space | 2024-11 | ok | wape_heating_pct=18.945 | epochs=7 | 37.7s
[400/700] U06 | h=1 | V21_core_heat_temp_space | 2024-12 | ok | 

,status,building,variant_id,horizon_hours,target_name,feature_count,feature_list,fold_id,train_end,test_start,test_end,architecture_key,architecture_label,lookback_hours,n_train_sequences,n_validation_sequences,n_test_sequences,n_test_heating_sequences,epochs_ran,best_val_loss,mae,rmse,wape_pct,r2,mae_heating,rmse_heating,wape_heating_pct,r2_heating,test_datetime_start,test_datetime_end
0,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-01,2023-12-31 23:00:00,2024-01-01 00:00:00,2024-01-31 23:00:00,single_64_l72,single 64 | L72,72,16378,1638,743,743,8,0.167550,26.875973,35.591445,19.073940,0.664819,26.875973,35.591445,19.073940,0.664819,2024-01-01 00:00:00,2024-01-31 22:00:00
1,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-02,2024-01-31 23:00:00,2024-02-01 00:00:00,2024-02-29 23:00:00,single_64_l72,single 64 | L72,72,17122,1712,695,695,7,0.282681,17.326250,23.107704,16.332191,0.786572,17.326250,23.107704,16.332191,0.786572,2024-02-01 00:00:00,2024-02-29 22:00:00
2,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-03,2024-02-29 23:00:00,2024-03-01 00:00:00,2024-03-31 23:00:00,single_64_l72,single 64 | L72,72,17818,1782,743,741,14,0.114560,11.279120,15.850045,14.629128,0.852212,11.277534,15.859379,14.589820,0.851179,2024-03-01 00:00:00,2024-03-31 22:00:00
3,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-11,2024-10-31 23:00:00,2024-11-01 00:00:00,2024-11-30 23:00:00,single_64_l72,single 64 | L72,72,23698,2370,719,719,9,0.035080,12.581583,19.278176,22.022785,0.821105,12.581583,19.278176,22.022785,0.821105,2024-11-01 00:00:00,2024-11-30 22:00:00
4,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-12,2024-11-30 23:00:00,2024-12-01 00:00:00,2024-12-31 23:00:00,single_64_l72,single 64 | L72,72,24418,2442,714,714,8,0.068241,13.890334,19.338157,16.644371,0.812994,13.890334,19.338157,16.644371,0.812994,2024-12-01 00:00:00,2024-12-30 17:00:00


,variant_id,feature_count,n_runs,mean_wape_heating_pct,std_wape_heating_pct,mean_rmse_heating,mean_mae_heating,mean_r2_heating,mean_delta_vs_core_pp,median_delta_vs_core_pp,win_rate_vs_core
0,V16_core_solar,9,50,18.916230,7.224034,24.359741,17.676662,0.642451,-0.047777,0.170277,0.44
1,V00_core,7,50,18.964007,7.733622,24.378546,17.552484,0.641974,0.000000,0.000000,0.00
2,V10_core_heat,13,50,19.097006,7.776257,24.758288,17.852626,0.638400,0.132999,0.127082,0.46
3,V11_core_temp,12,50,19.208251,8.006681,24.705660,17.904457,0.642538,0.244244,0.236992,0.46
4,V13_core_vent,12,50,19.261234,7.467196,25.051604,18.052392,0.628198,0.297227,0.151307,0.42
5,V22_core_heat_temp_vent,23,50,19.294907,7.828943,25.107742,18.123292,0.628763,0.330900,0.291100,0.42
6,V23_core_heat_temp_space_vent,28,50,19.332659,7.926584,24.925887,18.063117,0.637083,0.368652,0.425314,0.44
7,V20_core_heat_temp,18,50,19.480431,8.400290,25.032060,18.098268,0.631412,0.516424,0.134793,0.48
8,V14_core_dhw,12,50,19.553987,7.517788,25.218034,18.301309,0.622502,0.589980,0.741339,0.28
9,V21_core_heat_temp_space,23,50,19.612723,7.928193,25.423558,18.229860,0.615488,0.648717,0.407301,0.38


{'primary_metric': 'wape_heating_pct',
 'selection_rule': 'fallback_simplest_within_0p3_of_best',
 'selected_variant_id': 'V16_core_solar',
 'selected_feature_count': 9,
 'selected_mean_wape_heating_pct': 18.916229791641236,
 'selected_mean_delta_vs_core_pp': -0.04777686119079583,
 'selected_win_rate_vs_core': 0.44,
 'best_mean_wape_heating_pct': 18.916229791641236}

## Read Existing Results

This cell is safe to run even before a fresh screen. It will load prior outputs if they exist.

In [14]:
results_path = RESULTS_DIR / 'screen'
if results_path.exists() and (results_path / 'run_log.csv').exists():
    run_log_existing = pd.read_csv(results_path / 'run_log.csv')
    display(run_log_existing.head())
    if (results_path / 'variant_summary.csv').exists():
        variant_summary_existing = pd.read_csv(results_path / 'variant_summary.csv')
        display(variant_summary_existing)
    if (results_path / 'selection_summary.json').exists():
        selection_existing = json.loads((results_path / 'selection_summary.json').read_text())
        selection_existing
    else:
        print('No selection summary found yet.')
else:
    print('No screen results found yet. Run the screen cell above when ready.')

,status,building,variant_id,horizon_hours,target_name,feature_count,feature_list,fold_id,train_end,test_start,test_end,architecture_key,architecture_label,lookback_hours,n_train_sequences,n_validation_sequences,n_test_sequences,n_test_heating_sequences,epochs_ran,best_val_loss,mae,rmse,wape_pct,r2,mae_heating,rmse_heating,wape_heating_pct,r2_heating,test_datetime_start,test_datetime_end
0,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-01,2023-12-31 23:00:00,2024-01-01 00:00:00,2024-01-31 23:00:00,single_64_l72,single 64 | L72,72,16378,1638,743,743,8,0.167550,26.875973,35.591445,19.073940,0.664819,26.875973,35.591445,19.073940,0.664819,2024-01-01 00:00:00,2024-01-31 22:00:00
1,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-02,2024-01-31 23:00:00,2024-02-01 00:00:00,2024-02-29 23:00:00,single_64_l72,single 64 | L72,72,17122,1712,695,695,7,0.282681,17.326250,23.107704,16.332191,0.786572,17.326250,23.107704,16.332191,0.786572,2024-02-01 00:00:00,2024-02-29 22:00:00
2,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-03,2024-02-29 23:00:00,2024-03-01 00:00:00,2024-03-31 23:00:00,single_64_l72,single 64 | L72,72,17818,1782,743,741,14,0.114560,11.279120,15.850045,14.629128,0.852212,11.277534,15.859379,14.589820,0.851179,2024-03-01 00:00:00,2024-03-31 22:00:00
3,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-11,2024-10-31 23:00:00,2024-11-01 00:00:00,2024-11-30 23:00:00,single_64_l72,single 64 | L72,72,23698,2370,719,719,9,0.035080,12.581583,19.278176,22.022785,0.821105,12.581583,19.278176,22.022785,0.821105,2024-11-01 00:00:00,2024-11-30 22:00:00
4,ok,U05,V00_core,1,target_point_h1,7,"feat_heat_obs,feat_outdoor_temp_c,feat_hour_si...",2024-12,2024-11-30 23:00:00,2024-12-01 00:00:00,2024-12-31 23:00:00,single_64_l72,single 64 | L72,72,24418,2442,714,714,8,0.068241,13.890334,19.338157,16.644371,0.812994,13.890334,19.338157,16.644371,0.812994,2024-12-01 00:00:00,2024-12-30 17:00:00


,variant_id,feature_count,n_runs,mean_wape_heating_pct,std_wape_heating_pct,mean_rmse_heating,mean_mae_heating,mean_r2_heating,mean_delta_vs_core_pp,median_delta_vs_core_pp,win_rate_vs_core
0,V16_core_solar,9,50,18.916230,7.224034,24.359741,17.676662,0.642451,-0.047777,0.170277,0.44
1,V00_core,7,50,18.964007,7.733622,24.378546,17.552484,0.641974,0.000000,0.000000,0.00
2,V10_core_heat,13,50,19.097006,7.776257,24.758288,17.852626,0.638400,0.132999,0.127082,0.46
3,V11_core_temp,12,50,19.208251,8.006681,24.705660,17.904457,0.642538,0.244244,0.236992,0.46
4,V13_core_vent,12,50,19.261234,7.467196,25.051604,18.052392,0.628198,0.297227,0.151307,0.42
5,V22_core_heat_temp_vent,23,50,19.294907,7.828943,25.107742,18.123292,0.628763,0.330900,0.291100,0.42
6,V23_core_heat_temp_space_vent,28,50,19.332659,7.926584,24.925887,18.063117,0.637083,0.368652,0.425314,0.44
7,V20_core_heat_temp,18,50,19.480431,8.400290,25.032060,18.098268,0.631412,0.516424,0.134793,0.48
8,V14_core_dhw,12,50,19.553987,7.517788,25.218034,18.301309,0.622502,0.589980,0.741339,0.28
9,V21_core_heat_temp_space,23,50,19.612723,7.928193,25.423558,18.229860,0.615488,0.648717,0.407301,0.38


## How To Interpret The Outcome

A good thesis-quality result would look something like this:

- `core` is already strong,
- `heat_history` improves it,
- `temp_memory` helps a bit more,
- subsystem loops help selectively,
- `solar` hurts or is inconsistent,
- the largest dynamic variant is not the winner.

That would support a compact, operationally realistic feature story instead of a “throw all 70 features into the LSTM” story.